# 01 · Data exploration
What has been downloaded, how grid position turns into a result, and which features track the finish.
Run `python -m f1pred build` first.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from f1pred import build, config, evaluate, plotting, probabilities
from f1pred.features import FEATURE_SETS, FEATURES, build_features

plotting.apply_theme()
entries, laps, conditions = build.load_processed()
feats = build_features(entries, laps, conditions)
latest_season = int(feats.loc[feats["FinishPosition"].notna(), "Season"].max())

## Coverage

In [ ]:
log = pd.read_csv(config.LOG_PATH)
coverage = log.pivot_table(index="Season", columns="Status", values="SessionKey", aggfunc="count", fill_value=0)
coverage["Races built"] = entries.groupby("Season")["RoundNumber"].nunique()
display(coverage)

## Grid → finish

In [ ]:
rows = feats[(feats["Season"] == latest_season) & feats["FinishPosition"].notna()]
n = int(rows["NStarters"].max())
counts = pd.crosstab(rows["Grid"].astype(int), rows["FinishPosition"].astype(int)).reindex(
    index=range(1, n + 1), columns=range(1, n + 1), fill_value=0)

fig, ax = plt.subplots(figsize=(7.5, 6.5))
cmap = mpl.colors.LinearSegmentedColormap.from_list("blues", [plotting.SURFACE] + plotting.BLUES)
im = ax.imshow(counts.to_numpy(), cmap=cmap)
ticks = list(range(0, n, 2))
ax.set_xticks(ticks, [t + 1 for t in ticks])
ax.set_yticks(ticks, [t + 1 for t in ticks])
ax.set(title=f"{latest_season}: where each grid slot finished", xlabel="Finish position", ylabel="Grid position")
ax.grid(False)
bar = fig.colorbar(im, ax=ax, shrink=0.7, label="Races")
bar.outline.set_edgecolor(plotting.AXIS)
plt.show()

## Which features track the finish
Spearman ρ with finishing position (positive = higher value, worse finish).

In [ ]:
usable = [c for c in FEATURES if rows[c].nunique() > 1]
corr = pd.DataFrame({
    "All seasons": feats[usable].corrwith(feats["FinishPosition"], method="spearman"),
    str(latest_season): rows[usable].corrwith(rows["FinishPosition"], method="spearman"),
    "Missing %": feats[usable].isna().mean() * 100,
}).round(2).sort_values(str(latest_season), key=abs, ascending=False)
display(corr)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True)
panels = [("QGapPct", "Qualifying gap to pole (%)"), ("LongRunPct", "Long-run pace vs field median (%)")]
for ax, (col, label) in zip(axes, panels):
    d = rows[[col, "FinishPosition"]].dropna()
    d = d[d[col].between(d[col].quantile(0.01), d[col].quantile(0.99))]
    rho = d[col].corr(d["FinishPosition"], method="spearman")
    ax.scatter(d[col], d["FinishPosition"], s=36, color=plotting.ACCENT, alpha=0.6,
               edgecolors=plotting.SURFACE, linewidths=1)
    ax.set(title=f"{label}, ρ = {rho:.2f}", xlabel=label)
axes[0].set_ylabel("Finish position")
axes[0].set_yticks([1, 5, 10, 15, 20])
axes[0].set_ylim(rows["FinishPosition"].max() + 0.5, 0.5)
plt.show()

## DNFs

In [ ]:
started = feats[feats["FinishPosition"].notna()]
dnf = started["Classified"].astype("string").isin(["R", "D", "E", "N", "F"]).groupby(started["Season"]).mean()
display((dnf * 100).round(1).rename("DNF %").to_frame())